# Kev-0.8B GPU single-request latency on LANESHIFT observations

This notebook measures **warm, single-request** latency of the released
`jaredpalmer/kev-0.8b@v1.0` checkpoint on a Colab GPU using the actual scene
prompts in `benchmark_requests.jsonl`.

- One request at a time; no batch throughput.
- Five warm-up requests, then one timed pass over the fixture.
- Reports wall-clock p50/p95 and the model's own `latency_ms`.
- In-process only. It does **not** start `kev.serve` or expose an API.
- The CPU rows are the measured reference from the LANESHIFT host; they are not
  a controlled same-machine comparison.


## 1. Runtime

Choose **Runtime > Change runtime type > T4 GPU** (or L4/A100). The notebook
requires Python 3.12-3.13, which the Kev repository declares. If Colab gives an
older Python, switch to a runtime with 3.12+ before installing; a venv created
inside the notebook is not the kernel Python.


In [ ]:
!nvidia-smi
import sys, torch
print("python", sys.version)
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
    print("bf16_supported", torch.cuda.is_bf16_supported())


## 2. Install Kev

This installs the upstream training/serving package into the notebook kernel.
It does not download the base model yet; the checkpoint load below does that.


In [ ]:
!git clone --depth 1 https://github.com/jaredpalmer/kev.git /content/kev
%cd /content/kev
%pip install -q -e .


## 3. Upload the observation fixture

Upload `benchmark_requests.jsonl` from the LANESHIFT `notebooks/` directory. It
contains 24 actual schema-2 LANESHIFT decisions, rendered with the production
scene prompt. The reference label is included only for provenance; this
notebook measures latency, not accuracy.


In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
name = next(iter(uploaded))
REQUESTS = Path("/content/benchmark_requests.jsonl")
REQUESTS.write_bytes(uploaded[name])
print("wrote", REQUESTS, REQUESTS.stat().st_size, "bytes")


## 4. Inspect the actual requests

This confirms the fixture is real LANESHIFT data and that every prompt fits the
L256 budget before any timing.


In [ ]:
import json
from pathlib import Path
rows = [json.loads(line) for line in Path("/content/benchmark_requests.jsonl").read_text().splitlines() if line.strip()]
print("requests", len(rows))
print("sources", sorted({r.get("source") for r in rows}))
print("seeds", sorted({r.get("scenario_seed") for r in rows}))
print("total tokens min/max", min(r["laneshift_meta"]["total_tokens"] for r in rows), max(r["laneshift_meta"]["total_tokens"] for r in rows))
print("rendered vehicles", sorted({r["laneshift_meta"].get("vehicles_rendered") for r in rows}))
for r in rows[:3]:
    print({k: r.get(k) for k in ("id", "source", "scenario_seed", "label_for_reference")},
          "tokens", r["laneshift_meta"]["total_tokens"], "prob-style state chars", len(r["state"]))


## 5. Load the checkpoint and warm up

`LocalPredictor` scores one record in-process. It applies the checkpoint's
temperature and returns probabilities, logits and a model-time `latency_ms`.
The load is timed separately from the request loop.


In [ ]:
import time, torch
from kev.checkpoint import LoadOptions
from kev.predictors import LocalPredictor

if not torch.cuda.is_available():
    raise SystemExit("No GPU. Change the Colab runtime to T4/L4/A100.")

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("loading dtype", dtype)
torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
predictor = LocalPredictor(
    "jaredpalmer/kev-0.8b@v1.0",
    "cuda",
    LoadOptions(dtype=dtype, merge=True),
)
load_s = time.perf_counter() - t0
print(f"load_s {load_s:.2f}")
print(f"peak_vram_gb {torch.cuda.max_memory_allocated() / 1e9:.2f}")

from kev.data import load_records
records = load_records("/content/benchmark_requests.jsonl")
for record in records[:5]:
    predictor(record)
torch.cuda.synchronize()
print("warm-up complete")


## 6. Measure warm single-request latency

Each iteration calls the predictor once, synchronizes the GPU, and records the
wall time. No requests are batched or overlapped. The model's own `latency_ms`
is recorded alongside the wall time so transport/encoding overhead is visible.


In [ ]:
import json, statistics, time, torch
from pathlib import Path

wall_ms, model_ms, tokens = [], [], []
for record in records:
    torch.cuda.synchronize()
    start = time.perf_counter()
    out = predictor(record)
    torch.cuda.synchronize()
    wall_ms.append((time.perf_counter() - start) * 1000.0)
    model_ms.append(float(out["latency_ms"]))
    tokens.append(int(out["input_tokens"]))

def pct(values, fraction):
    ordered = sorted(values)
    index = min(len(ordered) - 1, max(0, int(round(fraction * (len(ordered) - 1)))))
    return ordered[index]

summary = {
    "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "dtype": str(dtype).replace("torch.", ""),
    "requests": len(records),
    "load_s": round(load_s, 3),
    "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 3),
    "input_tokens_min": min(tokens),
    "input_tokens_max": max(tokens),
    "wall_ms_p50": round(pct(wall_ms, 0.50), 2),
    "wall_ms_p95": round(pct(wall_ms, 0.95), 2),
    "wall_ms_mean": round(statistics.fmean(wall_ms), 2),
    "wall_ms_min": round(min(wall_ms), 2),
    "wall_ms_max": round(max(wall_ms), 2),
    "model_ms_p50": round(pct(model_ms, 0.50), 2),
    "model_ms_p95": round(pct(model_ms, 0.95), 2),
    "model_ms_mean": round(statistics.fmean(model_ms), 2),
}
print(json.dumps(summary, indent=2))
Path("/content/benchmark_results.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")


## 7. Compare with the measured CPU reference

The CPU rows below are from the LANESHIFT host: native ARM64 and x64-emulated
ONNX Runtime at four threads, plus the documented VM live figure. They used the
same 0.8B checkpoint and a comparable scene state; they are reference points,
not a controlled cross-machine benchmark.


In [ ]:
cpu = [
    ("Native ARM64 CPU, 4 threads, warm", 1600.6, 1624.7),
    ("x64-emulated CPU, 4 threads", 2044.1, 2058.5),
    ("VM guest CPU, live (docs)", 2602.0, 2635.0),
]
print(f"{'device':42} {'p50 ms':>9} {'p95 ms':>9}")
print(f"{'Colab GPU, single request':42} {summary['wall_ms_p50']:9.1f} {summary['wall_ms_p95']:9.1f}")
for name, p50, p95 in cpu:
    print(f"{name:42} {p50:9.1f} {p95:9.1f}")


## 8. Save the result

Download the small JSON and keep it with the experiment record. Do not turn this
notebook into a hosted API; the measurements are the deliverable.


In [ ]:
from google.colab import files
files.download("/content/benchmark_results.json")
